# EDA 4: Latent Knowledge Hypothesis Testing

This notebook encodes Claude's market intuitions as testable hypotheses and validates them against historical data.

**Goal**: Transform latent market knowledge into validated, quantitative features.

## The 20 Latent Knowledge Hypotheses

1. **Sector Lead-Lag**: Semiconductors lead broader tech by 2-4 weeks
2. **Fed Day Drift**: Markets drift in direction of initial move post-FOMC
3. **Earnings Momentum**: Beats cluster within sectors and quarters
4. **Insider Timing**: Insiders buy 2-3 months before positive catalysts
5. **Options Flow Lead**: Unusual options activity precedes news by 1-3 days
6. **Retail Contrarian**: Extreme retail sentiment is contrarian signal
7. **Analyst Herding**: Analyst upgrades cluster and lag reality
8. **Buyback Timing**: Companies execute buybacks at local lows
9. **Gap Fade**: Opening gaps >3% fade 60%+ of the time
10. **Monday Reversal**: Friday momentum tends to reverse Monday
11. **FOMC Vol Crush**: Implied volatility collapses post-FOMC
12. **VIX Mean Reversion**: VIX >30 mean-reverts within 10 days
13. **Credit Leads Equity**: Credit stress precedes equity weakness
14. **Sector Rotation**: Defensive sectors outperform in late cycle
15. **Dollar Impact**: Strong dollar hurts emerging market exporters
16. **Energy-Inflation Link**: Energy prices lead inflation prints
17. **Banks-Rates Link**: Bank stocks lead in rising rate regimes
18. **Consumer-Retail**: Consumer confidence predicts retail sales
19. **Housing-Materials**: Housing starts predict materials demand
20. **Tech-VC Activity**: VC activity correlates with tech performance

In [ ]:
import sys
sys.path.insert(0, '/home/nock/projects/quant_suite')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from datetime import datetime, timedelta
from pathlib import Path
import yfinance as yf

from src.data.sources.yahoo import YahooDataSource

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]

In [ ]:
# Load data for testing hypotheses
yahoo = YahooDataSource()

# Key symbols for hypothesis testing
SYMBOLS = {
    'market': ['SPY', 'QQQ', 'IWM', 'DIA'],
    'sectors': ['XLK', 'XLF', 'XLE', 'XLV', 'XLY', 'XLP', 'XLU', 'XLB', 'XLI', 'XLRE'],
    'semis': ['SMH', 'SOXX', 'NVDA', 'AMD', 'INTC'],
    'banks': ['XLF', 'JPM', 'BAC', 'GS'],
    'volatility': ['^VIX'],
    'rates': ['^TNX'],  # 10-year yield
    'dollar': ['UUP'],  # Dollar ETF
}

all_symbols = [s for group in SYMBOLS.values() for s in group]
price_data = {}

for symbol in set(all_symbols):
    try:
        df = yahoo.get_historical_data(symbol.replace('^', ''), period="5y")
        if len(df) > 0:
            price_data[symbol] = df
            print(f"Loaded {symbol}: {len(df)} rows")
    except Exception as e:
        # Try yfinance directly for indices
        try:
            ticker = yf.Ticker(symbol)
            df = ticker.history(period="5y")
            df.columns = df.columns.str.lower()
            if len(df) > 0:
                price_data[symbol] = df
                print(f"Loaded {symbol} via yfinance: {len(df)} rows")
        except:
            print(f"Failed to load {symbol}")

## Hypothesis Testing Framework

In [ ]:
class HypothesisTest:
    """Framework for testing latent knowledge hypotheses."""
    
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
        self.result = None
    
    def compute_ic(self, feature: pd.Series, forward_returns: pd.Series) -> float:
        """Compute Information Coefficient."""
        aligned = pd.concat([feature, forward_returns], axis=1).dropna()
        if len(aligned) < 30:
            return np.nan
        return stats.spearmanr(aligned.iloc[:, 0], aligned.iloc[:, 1])[0]
    
    def compute_hit_rate(self, signal: pd.Series, forward_returns: pd.Series) -> float:
        """Compute hit rate (directional accuracy)."""
        aligned = pd.concat([signal, forward_returns], axis=1).dropna()
        if len(aligned) < 30:
            return np.nan
        correct = (np.sign(aligned.iloc[:, 0]) == np.sign(aligned.iloc[:, 1])).mean()
        return correct
    
    def compute_sharpe(self, returns: pd.Series) -> float:
        """Compute annualized Sharpe ratio."""
        if len(returns.dropna()) < 30:
            return np.nan
        return returns.mean() / returns.std() * np.sqrt(252)
    
    def summarize(self, ic: float, hit_rate: float, sharpe: float = None, n: int = 0) -> dict:
        """Summarize test results."""
        validated = abs(ic) > 0.02 if not np.isnan(ic) else False
        
        self.result = {
            'name': self.name,
            'description': self.description,
            'ic': ic,
            'hit_rate': hit_rate,
            'sharpe': sharpe,
            'n_observations': n,
            'validated': validated,
            'grade': self._grade(ic)
        }
        return self.result
    
    def _grade(self, ic: float) -> str:
        abs_ic = abs(ic) if not np.isnan(ic) else 0
        if abs_ic >= 0.05:
            return 'A'
        elif abs_ic >= 0.03:
            return 'B'
        elif abs_ic >= 0.02:
            return 'C'
        elif abs_ic >= 0.01:
            return 'D'
        return 'F'

print("Hypothesis testing framework initialized.")

## Hypothesis 1: Sector Lead-Lag

**Claim**: Semiconductors lead broader tech by 2-4 weeks

In [ ]:
# Test: Semis lead tech
test = HypothesisTest('semis_lead_tech', 'Semiconductors lead broader tech by 2-4 weeks')

if 'SMH' in price_data and 'XLK' in price_data:
    smh = price_data['SMH']['close']
    xlk = price_data['XLK']['close']
    
    # SMH returns as feature, XLK future returns as target
    smh_returns = smh.pct_change(5)  # 5-day momentum
    xlk_fwd_returns = xlk.shift(-10) / xlk - 1  # 10-day forward
    
    ic = test.compute_ic(smh_returns, xlk_fwd_returns)
    hit_rate = test.compute_hit_rate(smh_returns, xlk_fwd_returns)
    n = len(pd.concat([smh_returns, xlk_fwd_returns], axis=1).dropna())
    
    result = test.summarize(ic, hit_rate, n=n)
    print(f"Hypothesis: {test.description}")
    print(f"IC: {ic:.4f}, Hit Rate: {hit_rate:.2%}, N: {n}")
    print(f"Validated: {result['validated']}, Grade: {result['grade']}")
else:
    print("Missing data for semis/tech test")

In [ ]:
# Visualize lead-lag relationship
if 'SMH' in price_data and 'XLK' in price_data:
    # Cross-correlation analysis
    smh_ret = price_data['SMH']['close'].pct_change()
    xlk_ret = price_data['XLK']['close'].pct_change()
    
    lags = range(-20, 21)
    correlations = []
    for lag in lags:
        if lag < 0:
            corr = smh_ret.iloc[-lag:].corr(xlk_ret.iloc[:lag])
        elif lag > 0:
            corr = smh_ret.iloc[:-lag].corr(xlk_ret.iloc[lag:])
        else:
            corr = smh_ret.corr(xlk_ret)
        correlations.append(corr)
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(lags, correlations, color='steelblue', alpha=0.7)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    ax.set_xlabel('Lag (days) - Positive = SMH leads XLK')
    ax.set_ylabel('Correlation')
    ax.set_title('Cross-Correlation: SMH vs XLK Returns')
    plt.tight_layout()
    plt.show()

## Hypothesis 9: Gap Fade

**Claim**: Opening gaps >3% fade 60%+ of the time

In [ ]:
# Test: Gap fade effect
test = HypothesisTest('gap_fade', 'Opening gaps >3% fade 60%+ of the time')

if 'SPY' in price_data:
    spy = price_data['SPY'].copy()
    
    # Calculate opening gap
    spy['gap'] = (spy['open'] - spy['close'].shift(1)) / spy['close'].shift(1)
    spy['intraday_return'] = (spy['close'] - spy['open']) / spy['open']
    
    # Large gaps (>3% or <-3%)
    large_gap_up = spy['gap'] > 0.03
    large_gap_down = spy['gap'] < -0.03
    
    # Fade = intraday move in opposite direction
    gap_up_data = spy[large_gap_up]
    gap_down_data = spy[large_gap_down]
    
    if len(gap_up_data) > 10:
        fade_rate_up = (gap_up_data['intraday_return'] < 0).mean()
        print(f"Gap Up > 3%: {len(gap_up_data)} occurrences, Fade Rate: {fade_rate_up:.1%}")
    
    if len(gap_down_data) > 10:
        fade_rate_down = (gap_down_data['intraday_return'] > 0).mean()
        print(f"Gap Down > 3%: {len(gap_down_data)} occurrences, Fade Rate: {fade_rate_down:.1%}")
    
    # Overall fade rate
    large_gaps = spy[large_gap_up | large_gap_down]
    if len(large_gaps) > 10:
        fade_correct = np.where(large_gaps['gap'] > 0,
                                large_gaps['intraday_return'] < 0,
                                large_gaps['intraday_return'] > 0)
        overall_fade_rate = fade_correct.mean()
        print(f"\nOverall Fade Rate: {overall_fade_rate:.1%}")
        print(f"Validated: {overall_fade_rate > 0.55}")

## Hypothesis 10: Monday Reversal

**Claim**: Friday momentum tends to reverse Monday

In [ ]:
# Test: Monday reversal
test = HypothesisTest('monday_reversal', 'Friday momentum tends to reverse Monday')

if 'SPY' in price_data:
    spy = price_data['SPY'].copy()
    spy['dow'] = spy.index.dayofweek
    spy['return'] = spy['close'].pct_change()
    
    # Friday returns
    fridays = spy[spy['dow'] == 4]['return']
    
    # Monday returns (next trading day after Friday)
    mondays = spy[spy['dow'] == 0]['return']
    
    # Align Friday -> Monday
    friday_idx = spy[spy['dow'] == 4].index
    monday_idx = spy[spy['dow'] == 0].index
    
    pairs = []
    for fri in friday_idx:
        # Find next Monday
        next_mon = monday_idx[monday_idx > fri]
        if len(next_mon) > 0:
            mon = next_mon[0]
            if (mon - fri).days <= 4:  # Within expected range
                pairs.append((fri, mon))
    
    fri_returns = [spy.loc[p[0], 'return'] for p in pairs]
    mon_returns = [spy.loc[p[1], 'return'] for p in pairs]
    
    # Correlation between Friday and Monday returns
    corr = stats.spearmanr(fri_returns, mon_returns)[0]
    reversal_rate = (np.sign(fri_returns) != np.sign(mon_returns)).mean()
    
    print(f"Friday-Monday Correlation: {corr:.4f}")
    print(f"Reversal Rate: {reversal_rate:.1%}")
    print(f"Validated: {corr < -0.02 or reversal_rate > 0.52}")

## Hypothesis 12: VIX Mean Reversion

**Claim**: VIX >30 mean-reverts within 10 days

In [ ]:
# Test: VIX mean reversion
test = HypothesisTest('vix_mean_reversion', 'VIX >30 mean-reverts within 10 days')

if '^VIX' in price_data:
    vix = price_data['^VIX']['close']
    
    # Find VIX > 30 occurrences
    high_vix_days = vix[vix > 30].index
    
    # Measure VIX 10 days later
    mean_reversion_results = []
    for day in high_vix_days:
        future_idx = vix.index[vix.index > day]
        if len(future_idx) >= 10:
            vix_10d = vix.loc[future_idx[9]]
            vix_now = vix.loc[day]
            mean_reversion_results.append({
                'date': day,
                'vix_initial': vix_now,
                'vix_10d': vix_10d,
                'change': vix_10d - vix_now,
                'reverted': vix_10d < vix_now
            })
    
    if mean_reversion_results:
        mr_df = pd.DataFrame(mean_reversion_results)
        reversion_rate = mr_df['reverted'].mean()
        avg_change = mr_df['change'].mean()
        
        print(f"High VIX (>30) occurrences: {len(mr_df)}")
        print(f"Mean Reversion Rate: {reversion_rate:.1%}")
        print(f"Avg VIX Change (10d): {avg_change:.2f}")
        print(f"Validated: {reversion_rate > 0.6}")

## Hypothesis 17: Banks-Rates Link

**Claim**: Bank stocks lead in rising rate regimes

In [ ]:
# Test: Banks outperform when rates rise
test = HypothesisTest('banks_rates', 'Bank stocks lead in rising rate regimes')

if 'XLF' in price_data and '^TNX' in price_data:
    xlf = price_data['XLF']['close']
    tnx = price_data['^TNX']['close']
    spy = price_data.get('SPY', {}).get('close')
    
    # Align dates
    common_idx = xlf.index.intersection(tnx.index)
    if spy is not None:
        common_idx = common_idx.intersection(spy.index)
    
    xlf = xlf.loc[common_idx]
    tnx = tnx.loc[common_idx]
    if spy is not None:
        spy = spy.loc[common_idx]
    
    # Rate regime: rising vs falling
    rate_change_20d = tnx.pct_change(20)
    rising_rates = rate_change_20d > 0
    falling_rates = rate_change_20d < 0
    
    # XLF returns
    xlf_returns = xlf.pct_change(5)
    
    # XLF vs SPY relative performance
    if spy is not None:
        spy_returns = spy.pct_change(5)
        xlf_relative = xlf_returns - spy_returns
    else:
        xlf_relative = xlf_returns
    
    # Performance by regime
    xlf_rising = xlf_relative[rising_rates].mean() * 252
    xlf_falling = xlf_relative[falling_rates].mean() * 252
    
    print(f"XLF Relative Performance (annualized):")
    print(f"  Rising Rates: {xlf_rising:.2%}")
    print(f"  Falling Rates: {xlf_falling:.2%}")
    print(f"  Difference: {xlf_rising - xlf_falling:.2%}")
    print(f"Validated: {xlf_rising > xlf_falling}")

## Run All Hypothesis Tests

In [ ]:
# Define all hypotheses and run tests
all_results = []

# 1. Semis lead tech
if 'SMH' in price_data and 'XLK' in price_data:
    smh_ret = price_data['SMH']['close'].pct_change(5)
    xlk_fwd = price_data['XLK']['close'].shift(-10) / price_data['XLK']['close'] - 1
    test = HypothesisTest('semis_lead_tech', 'Semiconductors lead broader tech')
    ic = test.compute_ic(smh_ret, xlk_fwd)
    hr = test.compute_hit_rate(smh_ret, xlk_fwd)
    all_results.append(test.summarize(ic, hr))

# 9. Gap fade
if 'SPY' in price_data:
    spy = price_data['SPY'].copy()
    spy['gap'] = (spy['open'] - spy['close'].shift(1)) / spy['close'].shift(1)
    spy['intraday'] = (spy['close'] - spy['open']) / spy['open']
    large_gaps = spy[abs(spy['gap']) > 0.03]
    if len(large_gaps) > 10:
        fade_signal = -np.sign(large_gaps['gap'])
        hit_rate = (np.sign(fade_signal) == np.sign(large_gaps['intraday'])).mean()
        test = HypothesisTest('gap_fade', 'Opening gaps >3% fade 60%+ of the time')
        # Approximate IC as (hit_rate - 0.5) * 2
        ic = (hit_rate - 0.5) * 0.1  # Simplified
        all_results.append(test.summarize(ic, hit_rate))

# 10. Monday reversal
if 'SPY' in price_data:
    spy = price_data['SPY'].copy()
    spy['dow'] = spy.index.dayofweek
    spy['return'] = spy['close'].pct_change()
    friday_ret = spy[spy['dow'] == 4]['return'].values[:-1]
    monday_ret = spy[spy['dow'] == 0]['return'].values[1:]
    min_len = min(len(friday_ret), len(monday_ret))
    if min_len > 30:
        corr = stats.spearmanr(friday_ret[:min_len], monday_ret[:min_len])[0]
        reversal = (np.sign(friday_ret[:min_len]) != np.sign(monday_ret[:min_len])).mean()
        test = HypothesisTest('monday_reversal', 'Friday momentum reverses Monday')
        all_results.append(test.summarize(-corr, reversal))

# 12. VIX mean reversion
if '^VIX' in price_data:
    vix = price_data['^VIX']['close']
    high_vix = vix[vix > 30]
    if len(high_vix) > 10:
        vix_fwd_10d = vix.shift(-10)
        aligned = pd.concat([vix, vix_fwd_10d], axis=1).loc[high_vix.index].dropna()
        reversion = (aligned.iloc[:, 1] < aligned.iloc[:, 0]).mean()
        test = HypothesisTest('vix_mean_reversion', 'VIX >30 mean-reverts in 10 days')
        all_results.append(test.summarize(reversion - 0.5, reversion))

# 17. Banks-rates link
if 'XLF' in price_data and '^TNX' in price_data and 'SPY' in price_data:
    xlf = price_data['XLF']['close']
    tnx = price_data['^TNX']['close']
    spy = price_data['SPY']['close']
    common = xlf.index.intersection(tnx.index).intersection(spy.index)
    rate_change = tnx.loc[common].pct_change(20)
    xlf_rel = xlf.loc[common].pct_change(5) - spy.loc[common].pct_change(5)
    test = HypothesisTest('banks_rates', 'Banks outperform in rising rate regimes')
    ic = test.compute_ic(rate_change, xlf_rel.shift(-5))
    hr = test.compute_hit_rate(rate_change, xlf_rel.shift(-5))
    all_results.append(test.summarize(ic, hr))

print(f"Completed {len(all_results)} hypothesis tests")

In [ ]:
# Results summary
results_df = pd.DataFrame(all_results)
results_df = results_df.sort_values('ic', key=abs, ascending=False)

print("\n" + "="*80)
print("LATENT KNOWLEDGE HYPOTHESIS TEST RESULTS")
print("="*80)
results_df[['name', 'description', 'ic', 'hit_rate', 'validated', 'grade']]

In [ ]:
# Visualize results
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['green' if r['validated'] else 'red' for r in all_results]
names = [r['name'] for r in all_results]
ics = [r['ic'] if not np.isnan(r['ic']) else 0 for r in all_results]

ax.barh(names, ics, color=colors, alpha=0.7)
ax.axvline(0, color='black', linewidth=0.5)
ax.axvline(0.02, color='gray', linestyle='--', alpha=0.5, label='Validation threshold')
ax.axvline(-0.02, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Information Coefficient')
ax.set_title('Latent Knowledge Hypothesis Validation')
ax.legend()

plt.tight_layout()
plt.show()

## Summary & Next Steps

### Validated Hypotheses

The following hypotheses have been validated (IC > 0.02 or hit rate > 55%):

In [ ]:
validated = results_df[results_df['validated'] == True]
not_validated = results_df[results_df['validated'] == False]

print(f"VALIDATED ({len(validated)}/{len(results_df)}):")
for _, row in validated.iterrows():
    print(f"  - {row['name']}: IC={row['ic']:.4f}, Hit Rate={row['hit_rate']:.1%}")

print(f"\nNOT VALIDATED ({len(not_validated)}/{len(results_df)}):")
for _, row in not_validated.iterrows():
    print(f"  - {row['name']}: IC={row['ic']:.4f}")

In [ ]:
# Save results
import json

# Convert results to serializable format
output_results = []
for r in all_results:
    output_results.append({
        'name': r['name'],
        'description': r['description'],
        'ic': float(r['ic']) if not np.isnan(r['ic']) else None,
        'hit_rate': float(r['hit_rate']) if not np.isnan(r['hit_rate']) else None,
        'validated': bool(r['validated']),
        'grade': r['grade']
    })

summary = {
    'timestamp': datetime.now().isoformat(),
    'total_hypotheses': len(all_results),
    'validated_count': len([r for r in all_results if r['validated']]),
    'results': output_results,
    'recommendations': [
        'Implement validated hypotheses as features',
        'Run more rigorous tests with longer history',
        'Test regime-conditional versions of failed hypotheses',
        'Consider combining multiple weak signals into composite'
    ]
}

output_path = Path("/home/nock/quant_results/live/research/latent_knowledge_tests.json")
with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Results saved to {output_path}")

### Next Steps

1. **Implement validated hypotheses as features** in `src/data/feature_engineering/latent_knowledge_features.py`
2. **Test additional hypotheses** from the full list of 20
3. **Run regime-conditional tests** for hypotheses that may work only in specific conditions
4. **Combine weak signals** into composite features
5. **Proceed to Phase 2** (Feature Engineering) to encode these insights